# nnUNetV2 Kidney Abnormality Segmentation — Training Notebook

**Dataset:** Dataset500_KidneyAbnormalities (290 cases)
**Task:** Multi-class segmentation (kidney, cyst, stones, tumors)
**Configuration:** 3D Full Resolution
**Environment:** Google Colab (GPU required)

---

## Instructions
1. Upload your `Dataset500_KidneyAbnormalities` folder to Google Drive
2. Update `DATASET_DRIVE_PATH` in **Cell 4** below
3. Run cells sequentially (don't skip)
4. If Colab disconnects during training, use the resume cell
5. After all folds complete, run the export cell to package your model

## Section 1: Runtime & Environment Setup

### Cell 1.1 — Check GPU Availability
Make sure you have a GPU (T4, L4, or A100). Free Colab = T4, Pro = L4/A100.

In [ ]:
!nvidia-smi

Mon Apr 27 17:19:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             51W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### Cell 1.2 — Install nnUNetV2
Takes ~2–3 minutes. Restart runtime if prompted after installation.

In [ ]:
!pip install nnunetv2 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.6/205.6 kB 7.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 76.9 MB/s eta 0:00:00


### Cell 1.3 — Mount Google Drive
You'll be asked to authorize access. Use the same Google account where your dataset is stored.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Section 2: Dataset Configuration

### Cell 2.1 — Set Your Google Drive Paths

**EDIT THESE TWO LINES:**
- `DATASET_DRIVE_PATH`: Folder containing `Dataset500_KidneyAbnormalities/`
- `RESULTS_DRIVE_PATH`: Where to save training results (checkpoints + predictions)

In [ ]:
# ============================================
# EDIT THESE PATHS TO MATCH YOUR DRIVE
# ============================================

# Example: if you uploaded to MyDrive/DATASET/
DATASET_DRIVE_PATH = "/content/drive/MyDrive/1THESIS_AMM/nnUNet_raw/Dataset500_KidneyAbnormalities"

# Where to back up results (will be created if it doesn't exist)
RESULTS_DRIVE_PATH = "/content/drive/MyDrive/1THESIS_AMM/nnUNet_raw/Dataset500_KidneyAbnormalities"

print("Dataset path:", DATASET_DRIVE_PATH)
print("Results backup path:", RESULTS_DRIVE_PATH)

Dataset path: /content/drive/MyDrive/1THESIS_AMM/nnUNet_raw/Dataset500_KidneyAbnormalities
Results backup path: /content/drive/MyDrive/1THESIS_AMM/nnUNet_raw/Dataset500_KidneyAbnormalities


### Cell 2.2 — Set nnUNet Environment Variables
These tell nnUNet where to read/write data during training.

In [ ]:
import os

os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/nnUNet_results"

print("nnUNet_raw:", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])
print("nnUNet_results:", os.environ["nnUNet_results"])

nnUNet_raw: /content/nnUNet_raw
nnUNet_preprocessed: /content/nnUNet_preprocessed
nnUNet_results: /content/nnUNet_results


### Cell 2.3 — Copy Dataset from Drive to Local Storage
Colab's local SSD is much faster than Google Drive for I/O-heavy training.

In [ ]:
import shutil
from pathlib import Path

# Create local folders
!mkdir -p /content/nnUNet_raw
!mkdir -p /content/nnUNet_preprocessed
!mkdir -p /content/nnUNet_results

# Copy dataset FROM Drive TO local nnUNet_raw
src = Path(DATASET_DRIVE_PATH)
dst = Path("/content/nnUNet_raw/Dataset500_KidneyAbnormalities")

if dst.exists():
    print("Dataset already exists locally. Skipping copy.")
else:
    print("Copying dataset from Drive...")
    shutil.copytree(src, dst)
    print("Done!")

# Verify local copy
images = list((dst / "imagesTr").glob("*.nii.gz"))
labels = list((dst / "labelsTr").glob("*.nii.gz"))
print(f"Images: {len(images)}")
print(f"Labels: {len(labels)}")
print(f"Match: {len(images) == len(labels)}")

Copying dataset from Drive...
Done!
Images: 290
Labels: 290
Match: True


## Section 3: Dataset Integrity Check
Quick sanity check before expensive preprocessing.

In [ ]:
import json
import nibabel as nib
import numpy as np

base = Path("/content/nnUNet_raw/Dataset500_KidneyAbnormalities")

# 1. Check dataset.json exists
dataset_json = base / "dataset.json"
assert dataset_json.exists(), "dataset.json not found!"
with open(dataset_json) as f:
    meta = json.load(f)
print("Dataset JSON loaded.")
print("Labels:", meta["labels"])
print("Channel names:", meta["channel_names"])

# 2. Quick spot check on 5 random labels
import random
lbl_files = list((base / "labelsTr").glob("*.nii.gz"))
sample = random.sample(lbl_files, min(5, len(lbl_files)))

issues = []
for p in sample:
    data = np.asanyarray(nib.load(p).dataobj)
    uniques = np.unique(data).astype(int)
    bad = [v for v in uniques if v not in [0,1,2,3,4]]
    if bad:
        issues.append((p.name, bad))
    print(f"  {p.name}: {uniques}")

if issues:
    print("\nISSUES FOUND:")
    for name, vals in issues:
        print(f"  {name}: bad values {vals}")
else:
    print("\nAll spot-checks passed!")

Dataset JSON loaded.
Labels: {'background': 0, 'kidney': 1, 'cyst': 2, 'stones': 3, 'tumors': 4}
Channel names: {'0': 'CT'}
  tumor_006.nii.gz: [0 1 4]
  stone_029.nii.gz: [0 1 3]
  multi_008.nii.gz: [0 1 2 4]
  multi_107.nii.gz: [0 1 2 4]
  stone_010.nii.gz: [0 1 3]

All spot-checks passed!


## Section 4: Planning & Preprocessing

This is the most important step. nnUNet will:
- Extract dataset fingerprint (sizes, spacings, intensities)
- Compute CT foreground percentiles for global normalization
- Design the U-Net architecture (patch size, depth, batch size)
- Preprocess all training cases

In [ ]:
!df -h

In [ ]:
!ps aux | grep -i nnunet | grep -v grep

In [ ]:
!du -sh /content/nnUNet_preprocessed/* 2>/dev/null || echo "No preprocessed folder yet"

In [ ]:
!df -h /tmp

In [ ]:
!rm -rf /content/nnUNet_preprocessed/Dataset500_KidneyAbnormalities/nnUNetPlans_2d
!rm -rf /content/nnUNet_preprocessed/Dataset500_KidneyAbnormalities/nnUNetPlans_3d_lowres
!rm -rf /content/nnUNet_preprocessed/Dataset500_KidneyAbnormalities/nnUNetPlans_3d_fullres

In [ ]:
!nnUNetv2_plan_and_preprocess -d 500 --verify_dataset_integrity -np 4

### Cell 4.1 — Backup Preprocessed Data & Plans to Drive

After planning/preprocessing finishes, immediately back up the preprocessed folder to Google Drive.
This prevents data loss if the Colab runtime disconnects before training starts.

In [ ]:
import shutil, json
from pathlib import Path

# ------------------------------------------------------------------
# 1. Check Drive path is valid
# ------------------------------------------------------------------
preprocessed_drive = Path(RESULTS_DRIVE_PATH).parent / "nnUNet_preprocessed"
print("Drive backup target:", preprocessed_drive)

try:
    preprocessed_drive.mkdir(parents=True, exist_ok=True)
    test_file = preprocessed_drive / ".write_test"
    test_file.write_text("ok")
    test_file.unlink()
    print("Drive path is valid and writable.\n")
except Exception as e:
    print("ERROR: Drive path is NOT accessible:", e)
    raise

# ------------------------------------------------------------------
# 2. Back up nnUNetPlans.json specifically (fast, most important)
# ------------------------------------------------------------------
local_plans = Path("/content/nnUNet_preprocessed/Dataset500_KidneyAbnormalities/nnUNetPlans.json")
drive_plans = preprocessed_drive / "Dataset500_KidneyAbnormalities" / "nnUNetPlans.json"

if local_plans.exists():
    drive_plans.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local_plans, drive_plans)
    print(f"Plans copied to Drive: {drive_plans}")
    if drive_plans.exists():
        print("VERIFIED: Plans file exists in Drive.\n")
    else:
        print("WARNING: Plans file copy verification FAILED.\n")
else:
    print("WARNING: Local plans not found. If preprocessing crashed, planning may not have completed.\n")

# ------------------------------------------------------------------
# 3. Back up entire preprocessed folder as tar (fast)
# ------------------------------------------------------------------
import subprocess
preprocessed_local = Path("/content/nnUNet_preprocessed")
tar_path = Path(RESULTS_DRIVE_PATH).parent / "nnUNet_preprocessed.tar"

if preprocessed_local.exists():
    print("Creating tar archive on Drive (faster than file-by-file copy)...")
    subprocess.run([
        "tar", "-cf", str(tar_path),
        "-C", "/content",
        "nnUNet_preprocessed"
    ], check=True)
    print(f"Backup complete: {tar_path}\n")
else:
    print("WARNING: Local preprocessed folder not found. Skipping full backup.\n")

# ------------------------------------------------------------------
# 4. Display plans locally
# ------------------------------------------------------------------
plans_display_path = local_plans if local_plans.exists() else drive_plans
if plans_display_path.exists():
    with open(plans_display_path) as f:
        plans = json.load(f)
    print("=== 3D Full-Res Configuration ===")
    print(json.dumps(plans["configurations"]["3d_fullres"], indent=2))
else:
    print("No plans file available to display.")

Drive backup target: /content/drive/MyDrive/1THESIS_AMM/nnUNet_raw/nnUNet_preprocessed
Drive path is valid and writable.


Creating tar archive on Drive (faster than file-by-file copy)...
Backup complete: /content/drive/MyDrive/1THESIS_AMM/nnUNet_raw/nnUNet_preprocessed.tar

=== 3D Full-Res Configuration ===
{
  "data_identifier": "nnUNetPlans_3d_fullres",
  "preprocessor_name": "DefaultPreprocessor",
  "batch_size": 2,
  "patch_size": [
    128,
    128,
    128
  ],
  "median_image_size_in_voxels": [
    434.5,
    512.0,
    467.0
  ],
  "spacing": [
    0.9130857586860657,
    0.78515625,
    0.81640625
  ],
  "normalization_schemes": [
    "CTNormalization"
  ],
  "use_mask_for_norm": [
    false
  ],
  "resampling_fn_data": "resample_data_or_seg_to_shape",
  "resampling_fn_seg": "resample_data_or_seg_to_shape",
  "resampling_fn_data_kwargs": {
    "is_seg": false,
    "order": 3,
    "order_z": 0,
    "force_separate_z": null
  },
  "resampling_fn_seg_kwargs": {
    "is_seg": tru

### (Optional) Inspect Generated Plans
You can view the automatically generated plans to see patch size, batch size, etc.

In [ ]:
from pathlib import Path

old_drive_folder = Path(RESULTS_DRIVE_PATH).parent / "nnUNet_preprocessed"

if old_drive_folder.exists():
    files = list(old_drive_folder.rglob("*"))
    print(f"Total files in old Drive backup: {len(files)}")
    # Show first 10
    for f in files[:10]:
        print(f)
else:
    print("Old Drive folder not found either.")

Total files in old Drive backup: 2908
/content/drive/MyDrive/1THESIS_AMM/nnUNet_raw/nnUNet_preprocessed/Dataset500_KidneyAbnormalities
/content/drive/MyDrive/1THESIS_AMM/nnUNet_raw/nnUNet_preprocessed/Dataset500_KidneyAbnormalities/dataset_fingerprint.json
/content/drive/MyDrive/1THESIS_AMM/nnUNet_raw/nnUNet_preprocessed/Dataset500_KidneyAbnormalities/dataset.json
/content/drive/MyDrive/1THESIS_AMM/nnUNet_raw/nnUNet_preprocessed/Dataset500_KidneyAbnormalities/nnUNetPlans.json
/content/drive/MyDrive/1THESIS_AMM/nnUNet_raw/nnUNet_preprocessed/Dataset500_KidneyAbnormalities/nnUNetPlans_3d_lowres
/content/drive/MyDrive/1THESIS_AMM/nnUNet_raw/nnUNet_preprocessed/Dataset500_KidneyAbnormalities/nnUNetPlans_2d
/content/drive/MyDrive/1THESIS_AMM/nnUNet_raw/nnUNet_preprocessed/Dataset500_KidneyAbnormalities/gt_segmentations
/content/drive/MyDrive/1THESIS_AMM/nnUNet_raw/nnUNet_preprocessed/Dataset500_KidneyAbnormalities/nnUNetPlans_3d_fullres
/content/drive/MyDrive/1THESIS_AMM/nnUNet_raw/nnUNet_p

In [ ]:
print("Extracting 64GB tar directly to Colab SSD. Please wait a few minutes...")

# Extract the tar file from Drive directly into the Colab root directory
!tar -xf /content/drive/MyDrive/1THESIS_AMM/nnUNet_raw/nnUNet_preprocessed_backup.tar -C /content/

print("✅ Extraction complete!")

# Let's verify the files are actually there!
!ls /content/nnUNet_preprocessed/Dataset500_KidneyAbnormalities

Extracting 64GB tar directly to Colab SSD. Please wait a few minutes...
✅ Extraction complete!
dataset_fingerprint.json  nnUNetPlans_2d	  nnUNetPlans.json
dataset.json		  nnUNetPlans_3d_fullres
gt_segmentations	  nnUNetPlans_3d_lowres


In [ ]:
import os

# Point to the freshly extracted local folder
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"

# Keep the raw and results folders pointing to Drive so your progress is saved
os.environ["nnUNet_raw"] = "/content/drive/MyDrive/1THESIS_AMM/nnUNet_raw"
os.environ["nnUNet_results"] = "/content/drive/MyDrive/1THESIS_AMM/nnUNet_results"

## Section 5: Model Training (3D Full-Res)

Each fold trains one model on 4/5 of the data.
**Train all 5 folds** for ensemble prediction (best performance).

**Time per fold (approx):**
- T4 GPU: ~4–6 hours
- L4 GPU: ~2–3 hours
- A100 GPU: ~1–2 hours

**Tip:** If Colab disconnects, re-run Section 1–2, then use the resume cell below.

### Cell 5.1 — Train Fold 0

In [ ]:
import json
with open("/content/nnUNet_preprocessed/Dataset500_KidneyAbnormalities/nnUNetPlans.json") as f:
    print("batch_size:", json.load(f)["configurations"]["3d_fullres"]["batch_size"])

batch_size: 2


In [ ]:
!nnUNetv2_train 500 3d_fullres 0 --npz --num_epochs 200

Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 995, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1133, in get_code
  File "<frozen importlib._bootstrap_external>", line 1063, in source_to_code
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/bin/nnUNetv2_train", line 5, in <module>
    from nnunetv2.run.run_training import run_training_entry
  File "/usr/local/lib/python3.12/dist-packages/nnunetv2/run/run_training.py", line 12, in <module>
    from nnunetv2.run.load_pretrained_weights import load_pretrained_weights
  File "/usr/local/lib/python3.12/dist-packages/nnunetv2/run/load_pretrained_weights.py", line 

### Cell 5.2 — Train Fold 1

In [ ]:
!nnUNetv2_train 500 3d_fullres 1 --npz --num_epochs 200


############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-04-27 08:12:21.977582: Using torch.compile...
2026-04-27 08:12:23.189332: do_dummy_2d_data_aug: False
2026-04-27 08:12:23.190449: Using splits from existing split file: /content/nnUNet_preprocessed/Dataset500_KidneyAbnormalities/splits_final.json
2026-04-27 08:12:23.190822: The split file c

### Cell 5.3 — Train Fold 2

In [ ]:
!nnUNetv2_train 500 3d_fullres 2 --npz --num_epochs 200

### Cell 5.4 — Train Fold 3

In [ ]:
!nnUNetv2_train 500 3d_fullres 3 --npz --num_epochs 200

### Cell 5.5 — Train Fold 4

In [ ]:
!nnUNetv2_train 500 3d_fullres 4 --npz --num_epochs 200

### Cell 5.6 — Resume Training (If Disconnected)

If Colab disconnects mid-training:
1. Re-run Cells 1.1–1.3, 2.2 (mount drive + set env vars)
2. Uncomment and run the cell below for the interrupted fold
3. The `--c` flag resumes from the last checkpoint

In [ ]:
# Uncomment the line for the fold you want to resume:

# !nnUNetv2_train 500 3d_fullres 0 --c --npz
# !nnUNetv2_train 500 3d_fullres 1 --c --npz
# !nnUNetv2_train 500 3d_fullres 2 --c --npz
# !nnUNetv2_train 500 3d_fullres 3 --c --npz
# !nnUNetv2_train 500 3d_fullres 4 --c --npz

print("Uncomment the fold you need to resume, then run this cell.")

## Section 6: Find Best Configuration

After all folds complete, nnUNet can tell you which configuration + postprocessing gives the best validation Dice score.

**Note:** This requires `--npz` flag during training (already included above).

In [ ]:
!nnUNetv2_find_best_configuration 500 -c 3d_fullres

## Section 7: 5-Fold Cross-Validation Evaluation

After all folds are trained, generate validation predictions for each fold and compute per-class metrics (DSC, IoU, HD95, Precision, Recall, F1-Score).
This produces the required `summary.json` for your thesis.

### Cell 7.1 — Generate Validation Predictions for All Folds

Reads `splits_final.json` to know which case belongs to which validation fold, runs `nnUNetv2_predict` fold-by-fold, and merges outputs.

In [ ]:
import os, json, subprocess, shutil
from pathlib import Path

dataset_id = 500
config = "3d_fullres"
preprocessed_dir = Path(f"/content/nnUNet_preprocessed/Dataset{dataset_id}_KidneyAbnormalities")
raw_dir = Path(f"/content/nnUNet_raw/Dataset{dataset_id}_KidneyAbnormalities")
results_dir = Path(f"/content/nnUNet_results/Dataset{dataset_id}_KidneyAbnormalities")

# Auto-detect trainer folder
trainer_candidates = list(results_dir.glob("nnUNetTrainer__*__3d_fullres"))
assert len(trainer_candidates) > 0, "No trained model found! Complete Section 5 first."
trainer_dir = trainer_candidates[0]
print("Trainer folder:", trainer_dir.name)

# Load splits
splits_file = preprocessed_dir / "splits_final.json"
assert splits_file.exists(), f"{splits_file} not found! Run planning/preprocessing first."
with open(splits_file) as f:
    splits = json.load(f)

cv_base = Path("/content/cv_predictions")
cv_base.mkdir(exist_ok=True)

for fold in range(5):
    print(f"\n=== Fold {fold} ===")
    val_cases = splits[fold]['val']
    print(f"Validation cases: {len(val_cases)}")

    fold_in = cv_base / f"fold_{fold}_input"
    fold_out = cv_base / f"fold_{fold}_output"
    fold_in.mkdir(exist_ok=True)
    fold_out.mkdir(exist_ok=True)

    # Clean old symlinks
    for p in list(fold_in.iterdir()):
        p.unlink()

    missing = 0
    for case in val_cases:
        src = raw_dir / "imagesTr" / f"{case}_0000.nii.gz"
        dst = fold_in / f"{case}_0000.nii.gz"
        if src.exists():
            if dst.exists():
                dst.unlink()
            dst.symlink_to(src)
        else:
            missing += 1
            print(f"  MISSING: {src.name}")
    if missing:
        print(f"  WARNING: {missing} cases missing, skipping them.")
        continue

    cmd = [
        "nnUNetv2_predict",
        "-i", str(fold_in),
        "-o", str(fold_out),
        "-d", str(dataset_id),
        "-c", config,
        "-f", str(fold),
        "-chk", "checkpoint_best.pth",
        "-npp", "1",
        "-nps", "1",
        "--verbose"
    ]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("ERROR!")
        print(result.stdout[-1000:])
        print(result.stderr[-1000:])
    else:
        print("Prediction complete.")

    # Merge into single folder
    merged = cv_base / "merged"
    merged.mkdir(exist_ok=True)
    for pred in fold_out.glob("*.nii.gz"):
        shutil.copy2(pred, merged / pred.name)

merged = cv_base / "merged"
print(f"\nAll validation predictions merged into: {merged}")
print(f"Total files: {len(list(merged.glob('*.nii.gz')))}")

### Cell 7.2 — Compute Per-Class Metrics & Save summary.json

Evaluates merged predictions against ground-truth labels.
Requires `medpy` and `scikit-learn`.

In [ ]:
!pip install medpy scikit-learn -q

In [ ]:
import json, numpy as np, nibabel as nib
from pathlib import Path
from tqdm.notebook import tqdm
from medpy.metric.binary import hd95
from sklearn.metrics import precision_score, recall_score, f1_score

pred_dir = Path("/content/cv_predictions/merged")
label_dir = Path("/content/nnUNet_raw/Dataset500_KidneyAbnormalities/labelsTr")
output_json = Path("/content/nnUNet_results/Dataset500_KidneyAbnormalities/cv_summary.json")
output_json.parent.mkdir(parents=True, exist_ok=True)

classes = [1, 2, 3, 4]
class_names = {1: "Kidney", 2: "Cyst", 3: "Stone", 4: "Tumor"}

results = {c: {"DSC": [], "IoU": [], "HD95": [], "Precision": [], "Recall": [], "F1": []} for c in classes}
per_case_results = []

pred_files = sorted(pred_dir.glob("*.nii.gz"))
print(f"Found {len(pred_files)} predictions.")

for pf in tqdm(pred_files):
    case_id = pf.name.replace(".nii.gz", "")
    lf = label_dir / f"{case_id}.nii.gz"
    if not lf.exists():
        print(f"Skipping {case_id}: label not found.")
        continue

    pred = np.asanyarray(nib.load(pf).dataobj).astype(np.uint8)
    label = np.asanyarray(nib.load(lf).dataobj).astype(np.uint8)

    case_entry = {"case_id": case_id}

    for c in classes:
        p = (pred == c).astype(np.uint8)
        l = (label == c).astype(np.uint8)

        if l.sum() == 0 and p.sum() == 0:
            dsc, iou, precision, recall, f1 = 1.0, 1.0, 1.0, 1.0, 1.0
            hd = None
        elif l.sum() == 0 or p.sum() == 0:
            dsc, iou, precision, recall, f1 = 0.0, 0.0, 0.0, 0.0, 0.0
            hd = None
        else:
            inter = np.logical_and(p, l).sum()
            union = np.logical_or(p, l).sum()
            dsc = float(2.0 * inter / (p.sum() + l.sum()))
            iou = float(inter / union)

            p_flat = p.flatten()
            l_flat = l.flatten()
            precision = float(precision_score(l_flat, p_flat, zero_division=0))
            recall = float(recall_score(l_flat, p_flat, zero_division=0))
            f1 = float(f1_score(l_flat, p_flat, zero_division=0))

            try:
                hd = float(hd95(p, l))
            except Exception as e:
                hd = None
                print(f"HD95 error {case_id} class {c}: {e}")

        results[c]["DSC"].append(dsc)
        results[c]["IoU"].append(iou)
        results[c]["HD95"].append(hd)
        results[c]["Precision"].append(precision)
        results[c]["Recall"].append(recall)
        results[c]["F1"].append(f1)

        case_entry[class_names[c]] = {
            "DSC": dsc, "IoU": iou, "HD95": hd,
            "Precision": precision, "Recall": recall, "F1": f1
        }

    per_case_results.append(case_entry)

# Aggregate
summary = {"per_case": per_case_results, "aggregate": {}}
for c in classes:
    agg = {}
    for metric, vals in results[c].items():
        clean = [v for v in vals if v is not None]
        if clean:
            agg[f"{metric}_mean"] = float(np.mean(clean))
            agg[f"{metric}_std"] = float(np.std(clean))
        else:
            agg[f"{metric}_mean"] = None
            agg[f"{metric}_std"] = None
    summary["aggregate"][class_names[c]] = agg

with open(output_json, "w") as f:
    json.dump(summary, f, indent=2)

print("\nSummary JSON saved to:", output_json)
print("\nAggregate Metrics:")
for cls_name, metrics in summary["aggregate"].items():
    print(f"  {cls_name}:")
    for k, v in metrics.items():
        if v is not None:
            print(f"    {k}: {v:.4f}")
        else:
            print(f"    {k}: N/A")

## Section 8: Custom Inference with Profiling

Run the full 5-fold ensemble on any volume(s) with per-volume latency and peak GPU VRAM logging.
Use this to benchmark computational efficiency for your thesis.

### Cell 8.1 — 5-Fold Ensemble Inference (Per-Volume Timing & VRAM)

Uses nnUNet's Python predictor API.
Edit `input_path` below to point to a single `.nii.gz` file or a folder of volumes.

In [ ]:
import torch
import json
import numpy as np
from pathlib import Path
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
from nnunetv2.imageio.simpleitk_reader_writer import SimpleITKIO

# -------------------- USER CONFIG --------------------
# Input: single file or folder containing .nii.gz volumes
input_path = Path("/content/nnUNet_raw/Dataset500_KidneyAbnormalities/imagesTr")  # USER: change this
output_folder = Path("/content/inference_output")
output_folder.mkdir(exist_ok=True)

# Which checkpoint to use ('checkpoint_best.pth' or 'checkpoint_final.pth')
checkpoint_name = "checkpoint_best.pth"
# -----------------------------------------------------

# Auto-detect model folder
results_dir = Path("/content/nnUNet_results/Dataset500_KidneyAbnormalities")
candidates = list(results_dir.glob("nnUNetTrainer__*__3d_fullres"))
if not candidates:
    raise RuntimeError("No trained model found in nnUNet_results!")
model_folder = candidates[0]
print("Auto-detected model folder:", model_folder.name)

# Initialize predictor
print("Loading ensemble (5 folds)...")
predictor = nnUNetPredictor(
    tile_step_size=0.5,
    use_gaussian=True,
    use_mirroring=True,
    device=torch.device('cuda', 0),
    verbose=False,
    verbose_preprocessing=False,
    allow_tqdm=True
)
predictor.initialize_from_trained_model_folder(
    str(model_folder),
    use_folds=(0, 1, 2, 3, 4),
    checkpoint_name=checkpoint_name
)
print("Predictor ready.\n")

# Determine files
if input_path.is_file():
    files = [input_path]
else:
    files = sorted(input_path.glob("*.nii.gz"))

if not files:
    raise ValueError(f"No .nii.gz files found in {input_path}")

io = SimpleITKIO()
profile = []

for f in files:
    print(f"Processing: {f.name}")

    # Read image
    image, props = io.read_images([str(f)])

    # Reset VRAM stats
    torch.cuda.reset_peak_memory_stats()

    # CUDA event timers
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    torch.cuda.synchronize()
    start_event.record()

    # Inference
    ret = predictor.predict_single_npy_array(image, props, None, None, False)
    segmentation = ret[0] if isinstance(ret, tuple) else ret

    end_event.record()
    torch.cuda.synchronize()

    latency_sec = start_event.elapsed_time(end_event) / 1000.0
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

    # Save prediction
    out_name = f.name.replace("_0000.nii.gz", ".nii.gz")
    if out_name == f.name:
        out_name = f.name
    out_file = output_folder / out_name
    io.write_seg(segmentation.astype(np.uint8), str(out_file), props)

    entry = {
        "file": f.name,
        "latency_seconds": round(latency_sec, 4),
        "peak_vram_gb": round(peak_vram_gb, 4)
    }
    profile.append(entry)
    print(f"  Latency: {latency_sec:.3f} s")
    print(f"  Peak VRAM: {peak_vram_gb:.2f} GB")
    print(f"  Saved: {out_file.name}\n")

# Save profiling log
profile_path = output_folder / "inference_profile.json"
with open(profile_path, "w") as f:
    json.dump(profile, f, indent=2)

print(f"All predictions saved to: {output_folder}")
print(f"Profiling log saved to: {profile_path}")

## Section 9: Save Results to Google Drive

Copy all training outputs (models, plans, validation predictions) to Drive for safekeeping.
**Does not modify** original `nnUNet_results/` — just copies.

In [ ]:
from pathlib import Path
import shutil

src = Path("/content/nnUNet_results")
dst = Path(RESULTS_DRIVE_PATH)

dst.mkdir(parents=True, exist_ok=True)

print("Copying nnUNet_results to Drive...")
for item in src.iterdir():
    dest_item = dst / item.name
    if item.is_dir():
        if dest_item.exists():
            shutil.rmtree(dest_item)
        shutil.copytree(item, dest_item)
    else:
        shutil.copy2(item, dest_item)
    print(f"  Copied: {item.name}")

print("\nAll results backed up to Drive!")

## Section 10: Export Model for Streamlit Deployment

Create a clean folder with **only the files needed** for your Streamlit app.
This is a **separate copy** — your original training outputs remain intact.

### Cell 10.1 — Create Streamlit Export Folder

In [ ]:
from pathlib import Path
import shutil
import json

# Paths
dataset_raw = Path("/content/nnUNet_raw/Dataset500_KidneyAbnormalities")
results_dir = Path("/content/nnUNet_results/Dataset500_KidneyAbnormalities")
export_dir = Path("/content/streamlit_export")

export_dir.mkdir(parents=True, exist_ok=True)

# 1. Copy dataset.json
shutil.copy2(dataset_raw / "dataset.json", export_dir / "dataset.json")
print("[1/4] Copied dataset.json")

# 2. Find and copy the best configuration folder
# nnUNet naming: nnUNetTrainer__nnUNetPlans__3d_fullres
trainer_folders = list(results_dir.glob("nnUNetTrainer__*__3d_fullres"))

if not trainer_folders:
    print("ERROR: No trained model found in nnUNet_results!")
else:
    trainer_folder = trainer_folders[0]
    export_trainer = export_dir / trainer_folder.name
    export_trainer.mkdir(exist_ok=True)

    # Copy dataset.json, dataset_fingerprint.json, plans.json
    for fname in ["dataset.json", "dataset_fingerprint.json", "plans.json"]:
        src = trainer_folder / fname
        if src.exists():
            shutil.copy2(src, export_trainer / fname)

    # Copy all fold folders
    for fold_dir in sorted(trainer_folder.glob("fold_*")):
        dst_fold = export_trainer / fold_dir.name
        if dst_fold.exists():
            shutil.rmtree(dst_fold)
        shutil.copytree(fold_dir, dst_fold)
        print(f"  Copied {fold_dir.name}")

    print(f"\n[2/4] Exported trainer: {trainer_folder.name}")

    # 3. Copy postprocessing file if it exists
    pp_file = results_dir / "postprocessing.pkl"
    if pp_file.exists():
        shutil.copy2(pp_file, export_dir / "postprocessing.pkl")
        print("[3/4] Copied postprocessing.pkl")
    else:
        print("[3/4] No postprocessing.pkl found (run find_best_configuration first)")

    # 4. Create README
    readme = export_dir / "README.txt"
    readme.write_text("""
nnUNetV2 Model Export for Streamlit
====================================
Dataset: Dataset500_KidneyAbnormalities
Configuration: 3d_fullres

Files needed in Streamlit app:
- dataset.json          (label names & channel info)
- plans.json            (network architecture config)
- fold_0/ to fold_4/    (model weights)
- postprocessing.pkl    (optional: best postprocessing)

Usage in Streamlit:
  MODEL_FOLDER = './models/Dataset500_KidneyAbnormalities/nnUNetTrainer__nnUNetPlans__3d_fullres'
  FOLD = 'all'  # or specific fold number
""")
    print("[4/4] Created README.txt")

print(f"\nExport complete: {export_dir}")
print("Contents:")
for item in sorted(export_dir.rglob("*")):
    if item.is_file():
        rel = item.relative_to(export_dir)
        size_mb = item.stat().st_size / (1024*1024)
        print(f"  {rel} ({size_mb:.1f} MB)")

### Cell 10.2 — Zip & Save to Drive

In [ ]:
import shutil
from pathlib import Path

zip_path = Path(RESULTS_DRIVE_PATH) / "streamlit_model_export.zip"
export_dir = Path("/content/streamlit_export")

# Create zip archive
shutil.make_archive(str(zip_path).replace(".zip", ""), 'zip', export_dir)

size_mb = zip_path.stat().st_size / (1024*1024)
print(f"Zip saved: {zip_path}")
print(f"Size: {size_mb:.1f} MB")
print("\nDownload this file and extract it into your Streamlit app's 'models/' folder.")

---

## End of Notebook

**Next Steps:**
1. Download `streamlit_model_export.zip` from your Drive
2. Extract it into your Streamlit project folder
3. Run `streamlit run app.py` (see separate `app.py` file)

**Need help?** Check the nnUNetV2 docs: https://github.com/MIC-DKFZ/nnUNet